# Study Tutor — Corte 1 Project

**AI Agentic Engineering · Ingeniería de Sistemas · Universidad de Santander**

Upload your class notes as a PDF and the tutor maps its topics, writes practice
questions grounded in the text, and grades your answers point by point.

## Architecture

```
                      student message
                            |
                     [ ORCHESTRATOR ]   routes and delegates; never answers itself
                            |
        +-------------------+-------------------+
        |                   |                   |
  Topic Extractor     Exam Generator         Grader
  reads ALL chunks    RAG on the topic       RAG on question + answer
  (no RAG - needs     + verbatim-quote       + per-criterion
   full coverage)       verification           verdicts
        |                   |                   |
        +-------------------+-------------------+
                            |
              ChromaDB  <-  ingestion (plain code, no LLM)
```

Agents live in this notebook. The deterministic plumbing — PDF reading, cleaning,
chunking, embeddings, vector store, retry policy — lives in `tutor/` where it is
unit-tested. See the README for setup.

## Running it

1. `pip install -r ../requirements.txt`
2. Run the setup cell; it creates `../.env` — paste your key from
   [AI Studio](https://aistudio.google.com/apikey).
3. Put a PDF in `../data/`.
4. **Run All.** Cells run top to bottom with no hidden state.

> Editing `.env` or `tutor/*.py` mid-session is handled: the setup cell enables
> `autoreload` and calls `config.reload()`. Re-run it after any change.


---
## 0. Setup


In [1]:
%pip install -q -r ../requirements.txt

Note: you may need to restart the kernel to use updated packages.


  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress t

In [1]:
# Pick up edits to tutor/*.py without restarting the kernel.
%load_ext autoreload
%autoreload 2

import logging
import sys
from pathlib import Path

# The notebook lives in notebooks/, the package one level up.
ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(ROOT))

from tutor import config   # importing this creates .env from .env.example if missing

# autoreload watches .py files, not .env - settings are re-read explicitly.
config.reload()

# Silence the SDK's automatic-function-calling notice; we use no tools.
logging.getLogger('google_genai.models').setLevel(logging.ERROR)

print('repo root      ', config.ROOT_DIR)
print('chat model     ', config.GEMINI_MODEL)
print('embed model    ', config.GEMINI_EMBED_MODEL, f'({config.EMBED_DIM} dims)')
print('chunk size     ', config.CHUNK_SIZE, 'chars, overlap', config.CHUNK_OVERLAP)
print('local fallback ', 'on' if config.ENABLE_LOCAL_FALLBACK else 'off (Gemini only)')
print('PDFs in data/  ', [p.name for p in config.DATA_DIR.glob('*.pdf')] or 'NONE - add one!')

# Fail here, not 20 cells later with a 404 that blames the model.
try:
    config.validate_models()
except Exception as error:
    print('\nCONFIGURATION PROBLEM:\n')
    print(error)

if config.GOOGLE_API_KEY:
    print('API key        loaded OK')
else:
    print(f'API key        MISSING -> open {config.ENV_FILE} and paste your key')


repo root       C:\Users\ACER\Desktop\AI_Agentic_Engineering_Project
chat model      gemini-3.6-flash
embed model     gemini-embedding-001 (768 dims)
chunk size      1200 chars, overlap 200
local fallback  off (Gemini only)
PDFs in data/   ['Test.pdf']
API key        loaded OK


---
## 1. Ingestion — deliberately not an agent

Reading a PDF, stripping headers, chunking and embedding is deterministic. There is no
judgement call, so an LLM here would only add cost and failure modes. It lives in
`tutor/ingest/` as plain, tested code.

Four decisions that make retrieval work:

| Decision | Why |
|---|---|
| Header/footer removal by position + repetition | A footer on 40 pages gets embedded 40 times and floods every result list. |
| Chunks respect paragraphs, 200-char overlap | A chunk cut mid-sentence embeds half an idea. |
| Chunk id = hash of its content | Re-ingesting overwrites instead of duplicating. |
| `RETRIEVAL_DOCUMENT` vs `RETRIEVAL_QUERY` | The embedding model is asymmetric; using one type for both hurts recall. |


In [2]:
from tutor.errors import TutorError
from tutor.ingest.pipeline import ingest
from tutor.vectorstore import get_collection

collection = get_collection()

if collection.count() > 0:
    print(f'Vector store already holds {collection.count()} chunks - skipping ingestion.')
    print('Call ingest(reset=True) if you replaced the PDFs in data/.')
else:
    try:
        stats = ingest()          # reads every PDF in data/
        for key, value in stats.items():
            print(f'{key:20} {value}')
    except TutorError as error:
        print('Ingestion could not run:\n')
        print(error)


  -> opening ChromaDB ...
  <- opening ChromaDB done in 0.2s
Vector store already holds 13 chunks - skipping ingestion.
Call ingest(reset=True) if you replaced the PDFs in data/.


### Retrieval check

Confirm retrieval finds the right material before building on it. Every agent inherits
whatever this returns.

**Read the scores.** If the top hits for a question you know the PDF answers sit below
~0.4, the chunking is wrong — fix it here.

First run is slow (ChromaDB loads, TLS handshake); later runs reuse both. Each stage
prints as it starts, so a silent pause means stuck, not slow — `python scripts/diagnose.py`
times each stage.


In [3]:
from tutor.vectorstore import search

QUESTION = '¿Por qué podemos considerar que las vacas son animales valiosos y beneficiosos para el ser humano?'

for hit in search(QUESTION, top_k=3):
    meta = hit['metadata']
    print(f"[{hit['score']:.3f}] {meta['source']} p.{meta['page']}")
    print('   ', hit['text'][:280].replace(chr(10), ' '), '...')
    print()


  cache hit on all 1 - no API call needed
  -> searching 13 chunks ...
  <- searching 13 chunks done in 0.0s
[0.791] Test.pdf p.7
    enos agrícolas, transportar materiales y proporcionar diversos recursos. También forman parte de numerosas culturas y pueden desempeñar un papel dentro de determinados sistemas ecológicos y agrícolas. Al mismo tiempo, reconocer su importancia implica reconocer nuestra responsabil ...

[0.776] Test.pdf p.4
    relación histórica entre humanos y bovinos ha sido mucho más amplia que simplemente obtener alimentos. Sin embargo, reconocer estos beneficios no significa ignorar los problemas relacionados con la ganadería. La producción bovina puede generar impactos ambientales importantes, es ...

[0.776] Test.pdf p.3
    Las vacas y su importancia para la humanidad Durante siglos, las vacas han proporcionado recursos fundamentales para las comunidades humanas. La leche, por ejemplo, ha sido utilizada para producir una enorme variedad de alimentos, entre ellos q

---
## 2. The LLM layer

Every agent calls `tutor.llm.generate()`. One retry policy, one structured-output
contract, one place to swap models.

Failures are classified, not lumped together:

| Failure | Response | Why |
|---|---|---|
| `429` quota | stop retrying | retrying cannot create quota |
| `503` / `504` / timeout | retry with backoff (2, 4, 8, 16s + jitter) | Google's shared models spike and recover |
| `4xx` | stop, wrapped with the API's own message | our config is wrong |
| anything else | re-raise raw | our bug, must be loud |

Falling back on *every* error would be worse than no fallback: a broken prompt would
get a quietly worse answer instead of a stack trace.

The local Qwen/Ollama fallback exists but is **off** (`ENABLE_LOCAL_FALLBACK=false`)
while building. Turn it on before the live demo.

> A `503` is Google's model being busy, not your code. Retries handle it.


In [ ]:
from tutor import llm

print('provider       ', config.LLM_PROVIDER)
print('chat model     ', config.active_chat_model())
print('embed model    ', config.active_embed_model(), f'({config.EMBED_PROVIDER})')
print('local fallback ', 'on' if config.ENABLE_LOCAL_FALLBACK else 'off')

needs_ollama = config.LLM_PROVIDER == 'ollama' or config.ENABLE_LOCAL_FALLBACK
if needs_ollama:
    ready = llm.ollama_available()
    print('ollama         ', 'ready' if ready else f'NOT reachable at {config.OLLAMA_HOST}')
    if not ready:
        print(f'                 run: ollama serve   and   ollama pull {config.OLLAMA_MODEL}')


### Switching provider mid-session

The Gemini free tier allows **20 requests per day per model**. When that runs out the
error says *"retry in 2.5s"* — ignore it, that is the per-minute `RetryInfo` attached to
every 429. A daily quota resets tomorrow.

Three ways out, no restart needed — edit `.env`, then re-run the setup cell:

| `.env` | Effect |
|---|---|
| `GEMINI_MODEL=gemini-3.5-flash-lite` | The quota is **per model**, so another one has its own 20. |
| `LLM_PROVIDER=ollama` | Local only. No Gemini call is attempted at all. |
| `ENABLE_LOCAL_FALLBACK=true` | Stay on Gemini, degrade to local automatically on 429. |

Or flip it from here for the rest of the session:


In [ ]:
# Uncomment to force the local model without touching .env.
# Everything downstream reads config.* at call time, so this takes effect immediately.

# config.LLM_PROVIDER = 'ollama'
# print('now using:', config.active_chat_model())

# Embeddings can move too, but the vector store must be rebuilt: vectors from
# different models are not comparable, and vectorstore refuses to mix them.
# config.EMBED_PROVIDER = 'ollama'
# ingest(reset=True)
pass


### Structured output, end to end

The Pydantic model is passed to `generate()` and used to validate the reply. This is
not "please answer in JSON" — the schema constrains decoding, so an invalid shape
cannot be produced. Every agent below uses this pattern.


In [5]:
from pydantic import BaseModel, Field

from tutor.vectorstore import search


class ChunkSanityCheck(BaseModel):
    """Throwaway schema: proves the plumbing works before we build agents."""

    language: str = Field(description='language of the text, e.g. Spanish')
    main_subject: str = Field(description='what this passage is about, in 5 words or fewer')
    is_exam_worthy: bool = Field(description='true if a question could be written from it')


sample = search('tema principal del documento', top_k=1)[0]

response = llm.generate(
    prompt=f'Analyse this passage from a student document:\n\n{sample["text"]}',
    system='You analyse study material. Answer only with the requested structure.',
    schema=ChunkSanityCheck,
)

print('answered by:', response.provider, '|', response.model)
print(response.parse(ChunkSanityCheck))


  cache hit on all 1 - no API call needed
  -> searching 13 chunks ...
  <- searching 13 chunks done in 0.0s
  -> gemini-3.6-flash ...
     still waiting on gemini-3.6-flash ... 5s
  <- gemini-3.6-flash failed after 9.8s
     Gemini busy (503 UNAVAILABLE. {'error': {'code': 503, 'message': 'This model is currently exp) - retry 1/4 in 2.3s
  -> gemini-3.6-flash (attempt 2/5) ...
     still waiting on gemini-3.6-flash (attempt 2/5) ... 5s
     still waiting on gemini-3.6-flash (attempt 2/5) ... 10s
     still waiting on gemini-3.6-flash (attempt 2/5) ... 15s
     still waiting on gemini-3.6-flash (attempt 2/5) ... 20s
     still waiting on gemini-3.6-flash (attempt 2/5) ... 25s
     still waiting on gemini-3.6-flash (attempt 2/5) ... 30s
     still waiting on gemini-3.6-flash (attempt 2/5) ... 35s
     still waiting on gemini-3.6-flash (attempt 2/5) ... 40s
     still waiting on gemini-3.6-flash (attempt 2/5) ... 45s
     still waiting on gemini-3.6-flash (attempt 2/5) ... 50s
     still

### The fallback drill (inactive)


In [6]:
# A safety net nobody tested is not one. Simulates a 429; needs ENABLE_LOCAL_FALLBACK=true.
if config.ENABLE_LOCAL_FALLBACK and llm.ollama_available():
    real_gemini = llm._call_gemini

    class SimulatedRateLimit(Exception):
        code = 429

    def rate_limited(*args, **kwargs):
        raise SimulatedRateLimit('429 RESOURCE_EXHAUSTED (simulated)')

    try:
        llm._call_gemini = rate_limited
        forced = llm.generate('Reply with the single word: alive.', temperature=0.0)
        print('answered by:', forced.provider, '|', forced.model)
    finally:
        llm._call_gemini = real_gemini   # restore even if the cell raises
else:
    print('Local fallback disabled - skipping the drill. This is expected right now.')


Local fallback disabled - skipping the drill. This is expected right now.


---
## 3. Agent 1 — Topic Extractor

Returns a structured map of the document: topics, subtopics, pages. Everything
downstream depends on it.

### Why this agent does NOT use RAG

The instinct is to embed "what are the topics?" and take the top 5. Two problems:
that phrase resembles nothing in particular, so the hits are noise; and topic
extraction needs **coverage**, while retrieval is built to discard most of the
document on purpose. Opposite goals.

So it reads everything, in document order. RAG earns its place in agents 2 and 3,
where the question really is "which passages are about *this*?".

### Map-reduce

13 chunks fit in one prompt; 200 do not. **Map**: extract topics per batch.
**Reduce**: a second call merges duplicates across batches. With one batch the reduce
is skipped — it could only cost quota and paraphrase the answer.


In [7]:
from pydantic import BaseModel, Field

from tutor import prompts
from tutor.vectorstore import list_all_chunks


class Topic(BaseModel):
    """One subject the document teaches."""

    name: str = Field(description='short topic name, in the language of the document')
    summary: str = Field(description='one or two sentences on what the document says about it')
    subtopics: list[str] = Field(description='specific ideas under this topic, 2 to 6 of them')
    source_pages: list[int] = Field(description='page numbers where this topic is discussed')


class TopicMap(BaseModel):
    topics: list[Topic]


print(prompts.load('system_prompt')[:300], '...')


You are a study tutor. You work only from the documents the student has uploaded:
their class notes, slides or textbook chapters. You help them prepare for an exam by
mapping what the material covers, writing practice questions, and telling them
specifically what their answers got wrong.

SCOPE
- Ev ...


The shared persona (`prompts/system_prompt.txt`) carries scope, refusal rules, language
and citation style; each agent adds only its own job on top. Composing them in
`prompts.system()` means no agent can run ungrounded by accident.


In [8]:
TOPIC_EXTRACTOR_ROLE = (
    'You are mapping what a study document covers, so the student knows what to revise.\n'
    'Read the passages and return the topics the document actually teaches.\n\n'
    'Rules:\n'
    '- A topic is something the document explains, not something it mentions in passing.\n'
    '- Use the document\'s own wording for names. It is the vocabulary the exam will use.\n'
    '- Merge near-duplicates into one topic rather than listing them separately.\n'
    '- source_pages must come from the [page N] markers, never invented.\n'
    '- Between 3 and 8 topics for a short document. Fewer, broader topics beat many thin ones.'
)

MERGE_ROLE = (
    'You are merging topic lists extracted from different parts of the same document.\n'
    'The same topic may appear more than once with different wording.\n\n'
    'Rules:\n'
    '- Merge topics that describe the same subject; keep the clearest name.\n'
    '- Union their subtopics and their source_pages; drop duplicates.\n'
    '- Do not invent topics that appear in none of the lists.'
)

print(prompts.system(TOPIC_EXTRACTOR_ROLE)[-500:])


ows what to revise.
Read the passages and return the topics the document actually teaches.

Rules:
- A topic is something the document explains, not something it mentions in passing.
- Use the document's own wording for names. It is the vocabulary the exam will use.
- Merge near-duplicates into one topic rather than listing them separately.
- source_pages must come from the [page N] markers, never invented.
- Between 3 and 8 topics for a short document. Fewer, broader topics beat many thin ones.


### The agent

Batches are sized in **characters, not chunks** — the limit that matters is the context
window. Each passage is labelled `[page N]`; without the marker the model would have to
guess page numbers, and it would.


In [9]:
from tutor import llm

BATCH_CHARS = 20_000   # fits the context window with room for the answer


def _batch_by_size(chunks, max_chars=BATCH_CHARS):
    """Group consecutive chunks into prompts that fit; order preserved."""
    batch, size = [], 0
    for chunk in chunks:
        if batch and size + len(chunk['text']) > max_chars:
            yield batch
            batch, size = [], 0
        batch.append(chunk)
        size += len(chunk['text'])
    if batch:
        yield batch


def _as_prompt(chunks):
    return '\n\n'.join(f"[page {c['metadata']['page']}] {c['text']}" for c in chunks)


def extract_topics(chunks=None) -> TopicMap:
    """Map over the document, then reduce the per-batch topic lists into one."""
    chunks = chunks if chunks is not None else list_all_chunks()
    if not chunks:
        raise ValueError('No chunks in the vector store - run the ingestion cell first.')

    batches = list(_batch_by_size(chunks))
    print(f'{len(chunks)} chunks -> {len(batches)} batch(es)')

    partials = []
    for number, batch in enumerate(batches, start=1):
        print(f'map {number}/{len(batches)}')
        response = llm.generate(
            prompt=_as_prompt(batch),
            system=prompts.system(TOPIC_EXTRACTOR_ROLE),
            schema=TopicMap,
        )
        partials.append(response.parse(TopicMap))

    if len(partials) == 1:
        return partials[0]   # nothing to merge; a reduce call could only paraphrase

    print('reduce')
    listing = '\n\n'.join(
        f'--- list {i} ---\n{partial.model_dump_json(indent=1)}'
        for i, partial in enumerate(partials, start=1)
    )
    merged = llm.generate(prompt=listing, system=prompts.system(MERGE_ROLE), schema=TopicMap)
    return merged.parse(TopicMap)


In [11]:
topic_map = extract_topics()

for topic in topic_map.topics:
    pages = ', '.join(str(p) for p in sorted(set(topic.source_pages)))
    print(f'\n{topic.name}   (p. {pages})')
    print(f'  {topic.summary}')
    for subtopic in topic.subtopics:
        print(f'    - {subtopic}')


13 chunks -> 1 batch(es)
map 1/1
  -> gemini-3.6-flash ...
     still waiting on gemini-3.6-flash ... 5s
     still waiting on gemini-3.6-flash ... 10s
     still waiting on gemini-3.6-flash ... 15s
     still waiting on gemini-3.6-flash ... 20s
     still waiting on gemini-3.6-flash ... 25s
     still waiting on gemini-3.6-flash ... 30s
     still waiting on gemini-3.6-flash ... 35s
     still waiting on gemini-3.6-flash ... 40s
     still waiting on gemini-3.6-flash ... 45s
     still waiting on gemini-3.6-flash ... 50s
     still waiting on gemini-3.6-flash ... 55s
  <- gemini-3.6-flash failed after 57.1s
     Gemini busy (504 DEADLINE_EXCEEDED. {'error': {'code': 504, 'message': 'Deadline expired befo) - retry 1/4 in 2.1s
  -> gemini-3.6-flash (attempt 2/5) ...
     still waiting on gemini-3.6-flash (attempt 2/5) ... 5s
     still waiting on gemini-3.6-flash (attempt 2/5) ... 10s
  <- gemini-3.6-flash (attempt 2/5) done in 12.0s

Temperamento y comportamiento social   (p. 1, 4, 5)


### Check the result against your PDF

The exam generator inherits every mistake here:

1. **Page numbers correct?** Open one and confirm.
2. **Anything missing?** An omitted topic will never be examined.
3. **The document's words or the model's?** A paraphrase trains you on the wrong terms.


---
## 4. Agent 2 — Exam Generator

**Here RAG is the right tool**: the question is now "which passages are about this
topic?", and we want a small focused subset.

The query uses the topic name **plus its subtopics**. The name alone retrieves too
narrowly to support four genuinely different questions.

Each question ships with `key_points` — what a correct answer must contain. The
generator had the passage in front of it, so it is the cheapest place to decide that.
The grader then checks against explicit criteria instead of re-deriving them: agents
passing structured state, not prose.

`difficulty` is recall / understanding / application, which tells the model something,
unlike easy/medium/hard.


In [12]:
from typing import Literal

from pydantic import BaseModel, Field

from tutor import grounding, llm, prompts
from tutor.vectorstore import search


class ExamQuestion(BaseModel):
    question: str = Field(description='the question, in the language of the document')
    expected_answer: str = Field(description='a complete correct answer, 2-4 sentences')
    key_points: list[str] = Field(description='2-4 ideas a correct answer must contain')
    difficulty: Literal['recall', 'understanding', 'application']
    source_page: int = Field(description='page this question comes from')
    source_quote: str = Field(
        description='one sentence copied VERBATIM from the passage that supports the answer'
    )


class Exam(BaseModel):
    topic: str
    questions: list[ExamQuestion]


In [13]:
EXAM_GENERATOR_ROLE = (
    'You write practice exam questions from a set of passages about one topic.\n\n'
    'Rules:\n'
    '- Every question must be answerable from the passages alone. If a passage does not\n'
    '  support a question, do not ask it.\n'
    '- source_quote must be copied CHARACTER FOR CHARACTER from a passage. Do not\n'
    '  paraphrase it, do not join two sentences, do not tidy it up. It is checked.\n'
    '- source_page must be the [page N] marker of the passage the quote came from.\n'
    '- key_points are the ideas an answer MUST contain to be correct - specific, not\n'
    '  vague restatements of the question.\n'
    '- Vary the questions. Four questions about the same sentence is one question.\n'
    '- Do not write multiple-choice options. These are open questions the student writes.'
)


### Then: verification

The prompt asks for a verbatim quote. Asking is not enough — a model under pressure to
supply a quote produces something quote-shaped, and it looks convincing.

So `tutor/grounding.py` checks each quote against the passages it came from. Fuzzy
enough to tolerate the accents and line breaks pypdf mangles, strict enough to reject a
sentence assembled from scattered words. Failures are flagged, not shipped.


In [14]:
RETRIEVAL_K = 6


def generate_exam(topic, n_questions: int = 4, top_k: int = RETRIEVAL_K) -> tuple[Exam, list[dict]]:
    """RAG over one topic, then structured generation. Returns the exam and its sources."""
    # Subtopics widen the query; the name alone retrieves too narrowly.
    query = f"{topic.name}. " + '. '.join(topic.subtopics)
    hits = search(query, top_k=top_k)
    if not hits:
        raise ValueError(f'No passages retrieved for topic {topic.name!r}.')

    passages = '\n\n'.join(f"[page {h['metadata']['page']}] {h['text']}" for h in hits)
    prompt = (
        f'Topic: {topic.name}\n'
        f'Subtopics: {", ".join(topic.subtopics)}\n\n'
        f'Write exactly {n_questions} questions from these passages:\n\n{passages}'
    )

    response = llm.generate(
        prompt=prompt,
        system=prompts.system(EXAM_GENERATOR_ROLE),
        schema=Exam,
        temperature=0.4,  # variety in phrasing, still faithful
    )
    return response.parse(Exam), hits


def verify_exam(exam: Exam, hits: list[dict]) -> list[dict]:
    """Check every source_quote against the passages it came from."""
    sources = [h['text'] for h in hits]
    pages = {h['metadata']['page'] for h in hits}
    report = []
    for index, question in enumerate(exam.questions, start=1):
        ok, score = grounding.is_grounded(question.source_quote, sources)
        report.append({
            'n': index,
            'grounded': ok,
            'score': score,
            'page_ok': question.source_page in pages,
        })
    return report


In [15]:
topic = topic_map.topics[0]
print('Topic:', topic.name, '\n')

exam, hits = generate_exam(topic, n_questions=4)
report = verify_exam(exam, hits)

for question, check in zip(exam.questions, report):
    flag = 'OK  ' if check['grounded'] else 'FLAG'
    print(f"\n[{flag}] Q{check['n']} ({question.difficulty}, p.{question.source_page}, "
          f"quote match {check['score']:.0%})")
    print(' ', question.question)
    print('   key points:', '; '.join(question.key_points))
    if not check['grounded']:
        print('   UNVERIFIED QUOTE:', question.source_quote[:160])
    if not check['page_ok']:
        print('   page not among the retrieved passages')

passed = sum(c['grounded'] and c['page_ok'] for c in report)
print(f"\n{passed}/{len(report)} questions traced to the document.")


Topic: Temperamento y comportamiento social 

  -> embedding 1 text(s) with gemini-embedding-001 ...
  <- embedding 1 text(s) with gemini-embedding-001 done in 0.5s
  -> searching 13 chunks ...
  <- searching 13 chunks done in 0.0s
  -> gemini-3.6-flash ...
     still waiting on gemini-3.6-flash ... 5s
     still waiting on gemini-3.6-flash ... 10s
     still waiting on gemini-3.6-flash ... 15s
     still waiting on gemini-3.6-flash ... 20s
     still waiting on gemini-3.6-flash ... 25s
     still waiting on gemini-3.6-flash ... 30s
     still waiting on gemini-3.6-flash ... 35s
     still waiting on gemini-3.6-flash ... 40s
     still waiting on gemini-3.6-flash ... 45s
     still waiting on gemini-3.6-flash ... 50s
     still waiting on gemini-3.6-flash ... 55s
  <- gemini-3.6-flash failed after 58.1s
     Gemini busy (504 DEADLINE_EXCEEDED. {'error': {'code': 504, 'message': 'Deadline expired befo) - retry 1/4 in 2.4s
  -> gemini-3.6-flash (attempt 2/5) ...
     still waiting on gem

### Reading the flags

- **85-100%** — grounded.
- **50-85%** — probably a real quote the model tidied up. Check it.
- **below 50%** — not in the document. Discard the question.
- **page not retrieved** — the quote may be real but the citation is wrong.

If most questions flag, the cause is usually upstream: chunks too small to hold a
quotable sentence.


---
## 5. Agent 3 — Grader

"Incorrect" teaches nothing. The student needs to know *which* idea they missed and
*where* it is explained.

**1. Grades against explicit criteria.** It walks the `key_points` and marks each
`covered` / `partial` / `missing`. Checking a list, not forming an impression.

**2. Retrieval runs on the student's answer too.** The question finds where the answer
*should* come from; the student's words find where their claims come from — or reveal
they come from nowhere in the document.

**3. The score is computed in code, never asked for.** A model asked for "a score out
of 10" is not reproducible and often contradicts its own words (*"you missed the main
point… 8/10"*). So the model only judges criteria and `tutor/scoring.py` does the
arithmetic. The score therefore cannot disagree with the feedback.


In [16]:
from typing import Literal

from pydantic import BaseModel, Field

from tutor import llm, prompts, scoring
from tutor.vectorstore import search


class PointVerdict(BaseModel):
    key_point: str = Field(description='the criterion being judged, copied as given')
    status: Literal['covered', 'partial', 'missing']
    comment: str = Field(description='one sentence: what the student said about this point')


class RawFeedback(BaseModel):
    """What the model returns. No score field: that is derived in code."""

    points: list[PointVerdict]
    what_was_right: str = Field(description='what the student did get right; empty if nothing')
    what_to_review: str = Field(description='the specific idea to go back and study')
    misconceptions: list[str] = Field(
        description='claims the student made that the document contradicts or never states'
    )
    source_pages: list[int] = Field(description='pages where the missing material is explained')


In [17]:
GRADER_ROLE = (
    'You are grading one written answer from a student against explicit criteria.\n\n'
    'Rules:\n'
    '- Judge every key point given to you, in the order given. Copy each one verbatim\n'
    '  into key_point so the student can see which criterion you are judging.\n'
    '- covered = the student stated this idea. partial = they gestured at it or got it\n'
    '  half right. missing = it is absent. Wording need not match; the idea must be there.\n'
    '- Be accurate, not encouraging. Marking a missing idea as covered because the answer\n'
    '  sounds confident is the worst thing you can do to someone preparing for an exam.\n'
    '- Name what was right first, and mean it. If nothing was right, say so plainly and\n'
    '  leave what_was_right empty rather than inventing praise.\n'
    '- misconceptions are claims the passages contradict or never make. An idea that is\n'
    '  simply absent from the answer is a missing point, not a misconception.\n'
    '- what_to_review names a specific idea and where to read it, never "review the topic".\n'
    '- Do not give a score or a grade. That is computed elsewhere.'
)


In [18]:
def grade_answer(question, student_answer: str, top_k: int = 5) -> dict:
    """Grade one answer; returns the judgements plus a derived score."""
    if not student_answer.strip():
        # No API call: the verdict is not in doubt.
        return {
            'feedback': RawFeedback(
                points=[PointVerdict(key_point=p, status='missing', comment='No answer given.')
                        for p in question.key_points],
                what_was_right='', what_to_review=question.expected_answer,
                misconceptions=[], source_pages=[question.source_page],
            ),
            'score': 0.0, 'verdict': 'incorrect',
        }

    # Both sides steer retrieval: where the answer should come from, and where
    # the student's claims come from.
    hits = search(f'{question.question} {student_answer}', top_k=top_k)
    passages = '\n\n'.join(f"[page {h['metadata']['page']}] {h['text']}" for h in hits)

    prompt = (
        f'QUESTION:\n{question.question}\n\n'
        f'KEY POINTS A CORRECT ANSWER MUST CONTAIN:\n'
        + '\n'.join(f'- {point}' for point in question.key_points)
        + f'\n\nREFERENCE ANSWER:\n{question.expected_answer}\n\n'
        f"STUDENT'S ANSWER:\n{student_answer}\n\n"
        f'PASSAGES FROM THEIR DOCUMENT:\n{passages}'
    )

    response = llm.generate(
        prompt=prompt,
        system=prompts.system(GRADER_ROLE),
        schema=RawFeedback,
        temperature=0.0,  # grading must be reproducible
    )
    feedback = response.parse(RawFeedback)

    score = scoring.score_from_points([p.status for p in feedback.points])
    return {'feedback': feedback, 'score': score, 'verdict': scoring.verdict_from_score(score)}


def show_feedback(question, result) -> None:
    feedback = result['feedback']
    print(f"{result['verdict'].replace('_', ' ').upper()}  ({result['score']:.0%})\n")
    for point in feedback.points:
        mark = {'covered': '+', 'partial': '~', 'missing': '-'}[point.status]
        print(f'  [{mark}] {point.key_point}')
        print(f'      {point.comment}')
    if feedback.what_was_right:
        print(f'\nRight: {feedback.what_was_right}')
    if feedback.misconceptions:
        print('\nIncorrect claims:')
        for item in feedback.misconceptions:
            print(f'  - {item}')
    pages = ', '.join(str(p) for p in sorted(set(feedback.source_pages)))
    print(f'\nReview: {feedback.what_to_review}  (p. {pages})')


### Three answers

Good, half-right, and confidently wrong. The third is the real test: fluent, plausible,
unsupported by the document. A grader that rewards confidence would pass it.


In [20]:
question = exam.questions[0]
print('Q:', question.question)
print('Key points:', '; '.join(question.key_points), '\n')

ANSWERS = {
    'good': question.expected_answer,
    'partial': 'Las vacas son animales tranquilos que pasan el día comiendo y descansando.',
    'confidently wrong': (
        'Las vacas son animales solitarios y territoriales que atacan a otros animales '
        'para defender su espacio, y por eso se crían siempre en corrales individuales.'
    ),
}

for label, answer in ANSWERS.items():
    print('\n' + '=' * 70)
    print(f'ANSWER ({label}): {answer[:110]}...')
    print('=' * 70)
    show_feedback(question, grade_answer(question, answer))


Q: ¿Qué actividades caracterizan el temperamento pacífico de las vacas en su vida cotidiana?
Key points: Dedican su tiempo a actividades pacíficas como alimentarse, descansar y caminar.; Interaccionan y se relacionan con otros miembros de su grupo.; No buscan atacar constantemente a otros seres vivos. 


ANSWER (good): En su vida cotidiana, las vacas dedican gran parte de su tiempo a actividades tranquilas e inofensivas. Entre ...
  cache hit on all 1 - no API call needed
  -> searching 13 chunks ...
  <- searching 13 chunks done in 0.0s
  -> gemini-3.6-flash ...
  <- gemini-3.6-flash failed after 4.8s
     Gemini busy (503 UNAVAILABLE. {'error': {'code': 503, 'message': 'This model is currently exp) - retry 1/4 in 2.4s
  -> gemini-3.6-flash (attempt 2/5) ...
     still waiting on gemini-3.6-flash (attempt 2/5) ... 5s
     still waiting on gemini-3.6-flash (attempt 2/5) ... 10s
     still waiting on gemini-3.6-flash (attempt 2/5) ... 15s
     still waiting on gemini-3.6-flash (attempt 

### What to check

- The **confidently wrong** answer must be `INCORRECT` with entries under *Incorrect
  claims*. If not, fix `GRADER_ROLE`, not the score.
- The **partial** answer should mix covered and missing. All-partial means the
  criteria are too vague.
- The percentage must match the `+ ~ -` marks — it is computed from them.


---
## 6. The Orchestrator

Something has to decide which agent a message is for. This layer is defined by what it
does **not** do: it never answers from the document itself. Every grounding guarantee
lives inside the specialists, so an orchestrator that answered directly would be an
unchecked fourth agent.

**Routing is structured output** — a `Literal` enforced during decoding, so an
invalid route cannot be returned. No `if 'examen' in message`.

**Arguments are resolved, never trusted.** The topic name the router extracts is a
model output, so it is matched against the real topic map before reaching retrieval.


In [21]:
from typing import Literal, Optional

from pydantic import BaseModel, Field

from tutor import grounding, llm, prompts
from tutor.session import StudySession


class Route(BaseModel):
    intent: Literal['map_topics', 'summarize_topic', 'generate_exam', 'grade_answer', 'progress', 'out_of_scope']
    topic: Optional[str] = Field(default=None, description='topic the student named, if any')
    n_questions: Optional[int] = Field(default=None, description='how many questions they asked for')


ROUTER_ROLE = (
    'You route a study request to the right tool. You do not answer it.\n\n'
    'Intents:\n'
    '- map_topics: they want to know what the document covers.\n'
    '- summarize_topic: they want an explanation or summary of one topic.\n'
    '- generate_exam: they want practice questions.\n'
    '- grade_answer: they are answering a question that is currently open.\n'
    '- progress: they ask how they are doing, or what to review.\n'
    '- out_of_scope: anything not about studying this document.\n\n'
    'Rules:\n'
    '- If a question is open and the message reads like an attempt to answer it, it is\n'
    '  grade_answer - even when it is short, hesitant, or wrong.\n'
    '- Copy the topic name as the student wrote it. Do not correct or translate it.\n'
    '- Never invent a topic that is not in the list you are given.'
)


In [22]:
def resolve_topic(name, topic_map):
    """Match a model-produced topic name against the real map.

    Fuzzy ('lo ambiental' -> 'Impacto ambiental y manejo del pastoreo'). Returns None
    rather than guessing: a wrong match examines the student on the wrong material.
    """
    if not name or topic_map is None:
        return None
    best, best_score = None, 0.0
    for topic in topic_map.topics:
        haystack = topic.name + ' ' + ' '.join(topic.subtopics)
        score = grounding.match_ratio(name, haystack)
        if score > best_score:
            best, best_score = topic, score
    return best if best_score >= 0.5 else None


def route(message: str, session: StudySession) -> Route:
    topics = [t.name for t in session.topic_map.topics] if session.topic_map else []
    pending = session.current_question
    context = (
        f'Topics available: {topics}\n'
        f'Open question: {pending.question if pending else "none"}\n\n'
        f'Conversation so far:\n{session.conversation.as_context() or "(new session)"}\n\n'
        f'Student message:\n{message}'
    )
    response = llm.generate(
        prompt=context,
        system=prompts.system(ROUTER_ROLE),
        schema=Route,
        temperature=0.0,  # routing is not creative work
    )
    return response.parse(Route)


### Dispatch and memory

`handle()` is a boring table from intent to agent call — all the judgement already
happened.

Memory is compacted **after** handling: sliding window of 6 turns plus a rolling
summary. Structured state (open exam, scores) lives in `StudySession`, never in the
transcript — which is what makes the window safe. It can forget turn 1 without
forgetting the exam.


In [23]:
def summarize_transcript(transcript: str) -> str:
    response = llm.generate(
        prompt=transcript,
        system=('Summarise this tutoring conversation in 3-5 bullet points: what the student '
                'studied, what they got wrong, and what they said about their own progress. '
                'Keep concrete details; drop pleasantries.'),
        temperature=0.0,
    )
    return response.text


def handle(message: str, session: StudySession) -> str:
    decision = route(message, session)
    session.conversation.add('student', message)
    reply_lines = [f'[route: {decision.intent}]']

    if decision.intent == 'map_topics':
        if session.topic_map is None:
            session.topic_map = extract_topics()
        for topic in session.topic_map.topics:
            reply_lines.append(f'- {topic.name}: {topic.summary}')

    elif decision.intent in ('summarize_topic', 'generate_exam'):
        topic = resolve_topic(decision.topic, session.topic_map)
        if topic is None:
            names = [t.name for t in session.topic_map.topics] if session.topic_map else []
            reply_lines.append(f'Which topic did you mean? Available: {names}')
        elif decision.intent == 'summarize_topic':
            reply_lines.append(f'{topic.name} (p. {sorted(set(topic.source_pages))})')
            reply_lines.append(topic.summary)
            reply_lines += [f'  - {s}' for s in topic.subtopics]
        else:
            new_exam, hits = generate_exam(topic, n_questions=decision.n_questions or 3)
            checks = verify_exam(new_exam, hits)
            # Only verified questions reach the student.
            new_exam.questions = [q for q, c in zip(new_exam.questions, checks) if c['grounded']]
            dropped = len(checks) - len(new_exam.questions)
            session.current_exam, session.current_index = new_exam, 0
            if dropped:
                reply_lines.append(f'({dropped} question(s) dropped: quote not found in the document)')
            question = session.current_question
            reply_lines.append(f'Question 1/{len(new_exam.questions)}: {question.question}')

    elif decision.intent == 'grade_answer':
        question = session.current_question
        if question is None:
            reply_lines.append('There is no open question. Ask me for a practice exam first.')
        else:
            result = grade_answer(question, message)
            topic_name = session.current_exam.topic
            session.record_score(topic_name, result['score'])
            session.advance()
            reply_lines.append(f"{result['verdict'].upper()} ({result['score']:.0%})")
            for point in result['feedback'].points:
                mark = {'covered': '+', 'partial': '~', 'missing': '-'}[point.status]
                reply_lines.append(f'  [{mark}] {point.key_point}')
            reply_lines.append(f"Review: {result['feedback'].what_to_review}")
            nxt = session.current_question
            if nxt:
                n = session.current_index + 1
                reply_lines.append(f'\nQuestion {n}/{len(session.current_exam.questions)}: {nxt.question}')
            else:
                reply_lines.append('\nThat was the last question of this topic.')

    elif decision.intent == 'progress':
        if not session.scores:
            reply_lines.append('You have not answered any questions yet.')
        else:
            for topic_name, average in session.weakest_topics():
                reply_lines.append(f'  {average:.0%}  {topic_name}')
            reply_lines.append('Weakest topic first - start your revision there.')

    else:
        reply_lines.append('I only work from the document you uploaded. Ask me about it.')

    reply = '\n'.join(reply_lines)
    session.conversation.add('tutor', reply)
    session.conversation.compact(summarize_transcript)
    return reply


### Full conversation

Nothing is scripted; every message goes through `handle()`. Watch the `[route: ...]`
tag — message 4 is a bare answer with no keywords and still reaches the grader,
because a question is open.


In [25]:
session = StudySession(topic_map=topic_map)

SCRIPT = [
    '¿De qué trata este documento?',
    'Explícame el tema del comportamiento social',
    'Hazme 2 preguntas de práctica sobre ese tema',
    'Las vacas viven solas y no forman grupos, cada una es independiente.',
    '¿Cómo voy?',
    '¿Cuál es la capital de Francia?',
]

for message in SCRIPT:
    print('\n' + '=' * 74)
    print('STUDENT:', message)
    print('-' * 74)
    print(handle(message, session))



STUDENT: ¿De qué trata este documento?
--------------------------------------------------------------------------
  -> gemini-3.6-flash ...
  <- gemini-3.6-flash failed after 0.4s


LLMError: Gemini rate limit reached (gemini-3.6-flash). This is quota, not a bug in your code.
Wait about a minute, or enable the local fallback with config.ENABLE_LOCAL_FALLBACK=true in .env.
Original error: 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. \n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 20, model: gemini-3.6-flash\nPlease retry in 2.57801372s.', 'status': 'RESOURCE_EXHAUSTED', 'details': [{'@type': 'type.googleapis.com/google.rpc.Help', 'links': [{'description': 'Learn more about Gemini API quotas', 'url': 'https://ai.google.dev/gemini-api/docs/rate-limits'}]}, {'@type': 'type.googleapis.com/google.rpc.QuotaFailure', 'violations': [{'quotaMetric': 'generativelan

### What this demonstrates

| | |
|---|---|
| **Sub-agents** | One router, three specialists, each with its own prompt, schema and retrieval strategy. |
| **RAG where it belongs** | Exam generator and grader; deliberately not the topic extractor. |
| **Context engineering** | Typed state + sliding window with rolling summary. |
| **Verification** | Quotes checked, ungrounded questions dropped, score derived from visible criteria. |
| **Refusal** | The last message routes to `out_of_scope`. |

Run the cell twice: the second run enters with the conversation already in memory and
routing still works.


---
## Appendix — secret leak check

This notebook is committed **with outputs** so it can be read on GitHub. A stray print
or traceback can therefore put the API key in the repo — `.gitignore` protects `.env`,
not this file.

Run after **Save**, before every commit.


In [ ]:
import json, re

nb_path = Path.cwd() / 'tutor.ipynb'
raw = nb_path.read_text(encoding='utf-8')

problems = []
if config.GOOGLE_API_KEY and config.GOOGLE_API_KEY in raw:
    problems.append('your GOOGLE_API_KEY appears verbatim in the saved notebook')
for match in set(re.findall(r'AIza[0-9A-Za-z_\-]{20,}', raw)):
    problems.append(f'a Google API key pattern appears: {match[:8]}...')

if problems:
    print('DO NOT COMMIT:')
    for problem in problems:
        print(' -', problem)
    print('\nClear the offending cell output (Cell > Current Outputs > Clear), save, re-run this check,')
    print('and rotate the key at aistudio.google.com if it was ever pushed.')
else:
    print('Clean: no API key found in the saved notebook. Safe to commit.')
